# Citi Bike 2024: where bikes are needed, and when

**Question.** Bike share has a daily operating problem: the morning commute carries bikes from
residential neighborhoods into office districts, and the evening commute carries them back.
Stations run empty or full, and crews move bikes by truck to fix it. This notebook asks:

1. How big and how regular is that imbalance?
2. How much do season and weather move demand?
3. Can tomorrow's demand be forecast station by station, well enough to plan rebalancing?

**Data.** All 44 million Citi Bike trips in 2024, NOAA's Central Park weather station, and NYC
Open Data's borough boundaries and bike lanes. Everything is downloaded and cached by
`scripts/pipeline.py`; the [live site](https://ancientabacus.github.io/NYC-CitiBike-Analysis/)
is built from the same scripts this notebook calls.

**How to run.** From the repo root, after `pip install -r requirements.txt`. The first run
downloads ~7.5 GB. The forecast itself takes ~45 minutes, so section 4 reads the results that
`scripts/build_forecast.py` saved rather than retraining; every other number is recomputed here.

In [ ]:
import json
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

sys.path.insert(0, "scripts")
import build_rebalancing
import build_seasonality
from pipeline import cleaning_stats, load_clean_period, load_weather_daily

YEAR = "2024"
BLUE, ORANGE, RED, GRAY = "#2a78d6", "#eb6834", "#e34948", "#8a8984"
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

## 1. Data and cleaning

Each monthly file is cleaned on its own by `pipeline.clean()`, in this order: duplicate ride ids,
trips that start outside the file's month, trips missing a station or coordinates, non-positive
durations, and trips over 24 hours (Citi Bike's lost/stolen threshold). Every analysis below,
and every section of the site, reads the same cleaned trips.

In [ ]:
stats = cleaning_stats(YEAR)
removed = pd.Series(stats["removed"], name="rows removed")
print(f"raw rows {stats['raw_rows']:,}  ->  clean {stats['clean_rows']:,}  "
      f"({1 - stats['clean_rows'] / stats['raw_rows']:.2%} removed)")
removed.to_frame()

In [ ]:
trips = load_clean_period(YEAR, columns=["started_at", "member_casual", "rideable_type", "trip_minutes"])
mix = pd.Series({
    "casual share": (trips["member_casual"] == "casual").mean(),
    "casual share, weekends": (trips.loc[trips["started_at"].dt.dayofweek >= 5, "member_casual"] == "casual").mean(),
    "e-bike share": (trips["rideable_type"] == "electric_bike").mean(),
}).map("{:.1%}".format)
display(mix.to_frame("2024"))
trips.groupby("member_casual", observed=True)["trip_minutes"].describe(percentiles=[0.5])[["mean", "50%"]].round(1)

## 2. Seasons and weather

Daily rides joined to NOAA's daily summaries, plus *hourly* precipitation so rain during the commute
can be separated from rain at 3am. The model (`build_seasonality.fit_weather_model`):

`log(rides) ~ day of week + holiday + tmax + tmax² + rain type + snow + growth trend`

Rain type is mutually exclusive versus a dry day: off-peak rain only, commute-hour rain, heavy
rain (10+ mm). Intervals come from a moving-block bootstrap (14-day blocks), because consecutive
days are correlated and resampling single days would understate the uncertainty.

In [ ]:
daily = (build_seasonality.daily_rides(YEAR)
         .merge(load_weather_daily(YEAR), on="date")
         .merge(build_seasonality.commute_precip(YEAR), on="date", how="left")
         .fillna({"commute_precip_mm": 0.0}))
model = build_seasonality.fit_weather_model(daily)
effects = pd.DataFrame({
    k: {"effect": v["effect"], "95% CI low": v["ci95"][0], "95% CI high": v["ci95"][1], "days": model["days_flagged"].get(k)}
    for k, v in model["effects"].items()
}).T
print(f"R² = {model['r2']}")
effects.style.format({"effect": "{:+.1%}", "95% CI low": "{:+.1%}", "95% CI high": "{:+.1%}", "days": "{:.0f}"}, na_rep="")

Read the rain rows together: **rain during commute hours costs about four times as much as
rain only at other times**. Snow's interval includes zero (only a handful of snow days), and the
growth trend's interval is wide, so neither is presented as a finding.

In [ ]:
fig, (a, b) = plt.subplots(1, 2, figsize=(12, 3.8), gridspec_kw={"width_ratios": [2, 1]})
a.plot(daily["date"], daily["rides"], color=BLUE, alpha=0.35, lw=1, label="daily rides")
a.plot(daily["date"], daily["rides"].rolling(7).mean(), color=BLUE, lw=2.2, label="7-day average")
wet = daily["commute_precip_mm"] >= 1
a.scatter(daily.loc[wet, "date"], daily.loc[wet, "rides"], s=10, color=ORANGE, zorder=3, label="rain at commute hours")
a.set_title("Rides per day, 2024"); a.legend(frameon=False, fontsize=8)

dry_weekday = (daily["date"].dt.dayofweek < 5) & (daily["precip_mm"] < 1)
bins = (daily["tmax_c"] // 5 * 5).astype(int)
binned = daily[dry_weekday].groupby(bins[dry_weekday])["rides"].agg(["mean", "size"])
binned = binned[binned["size"] >= 3]
b.bar(binned.index.astype(str) + "°", binned["mean"], color=BLUE)
b.set_title("Dry weekdays by high temperature")
plt.tight_layout(); plt.show()

## 3. The rebalancing problem

For each station on non-holiday weekdays: average arrivals minus departures between 7 and 10am
(and 4 to 7pm). Negative means the station loses bikes during the rush. "Bikes to move" is the
sum of every losing station's average deficit: a *lower bound* on the trucking needed, because
the trip data has no dock capacity and averaging hides day-to-day swings.

In [ ]:
flows_src = load_clean_period(YEAR, columns=["started_at", "ended_at", "start_station_name", "end_station_name"])
days = pd.Series(pd.to_datetime(flows_src["started_at"].dt.normalize().unique()))
n_weekdays = int(build_rebalancing.weekday_mask(days).sum())
wc = build_rebalancing.window_counts
flows = pd.DataFrame({
    "am_dep": wc(flows_src["started_at"], flows_src["start_station_name"], build_rebalancing.AM),
    "am_arr": wc(flows_src["ended_at"], flows_src["end_station_name"], build_rebalancing.AM),
    "pm_dep": wc(flows_src["started_at"], flows_src["start_station_name"], build_rebalancing.PM),
    "pm_arr": wc(flows_src["ended_at"], flows_src["end_station_name"], build_rebalancing.PM),
}).fillna(0) / n_weekdays
flows["am_net"] = flows["am_arr"] - flows["am_dep"]
flows["pm_net"] = flows["pm_arr"] - flows["pm_dep"]
active = flows[(flows["am_dep"] + flows["am_arr"]) >= 1]

am_share = active["am_net"] / (active["am_dep"] + active["am_arr"])
pm_share = active["pm_net"] / (active["pm_dep"] + active["pm_arr"])
print(f"{n_weekdays} weekdays, {len(active):,} active stations")
print(f"bikes to move back each weekday morning (lower bound): {(-active['am_net']).clip(lower=0).sum():,.0f}")
print(f"AM vs PM net flow correlation: raw {active['am_net'].corr(active['pm_net']):.2f}, "
      f"normalized by station traffic {am_share.corr(pm_share):.2f}")
active.nsmallest(8, "am_net")[["am_dep", "am_arr", "am_net", "pm_net"]].round(1)

The raw correlation is partly mechanical (busy stations have big flows in both directions),
so it's recomputed with each station's net flow divided by its own rush-hour traffic. It stays
strongly negative: the evening genuinely undoes the morning.

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 4.5))
ax.scatter(am_share, pm_share, s=6, alpha=0.4, color=BLUE)
ax.axhline(0, color=GRAY, lw=0.8); ax.axvline(0, color=GRAY, lw=0.8)
ax.set_xlabel("AM net flow / AM traffic"); ax.set_ylabel("PM net flow / PM traffic")
ax.set_title("Mornings and evenings mirror each other")
plt.tight_layout(); plt.show()
del flows_src

## 4. Forecasting demand

`scripts/build_forecast.py` trains two `HistGradientBoostingRegressor` models (Poisson loss), one
for departures and one for arrivals, at the station × hour grain (2,273 stations, ~19M rows).
Every feature is known a day ahead: hour, weekday, holiday, observed weather (an upper bound;
production would use forecast weather), lags of 24h and 168h, the same hour's 7-day mean, and
each station's training-period profile.

Evaluation is by date only: train Jan 8–Aug, early-stop on Sep–Oct, refit on Jan–Oct, test on
Nov–Dec. Two choices worth calling out:

- **The baseline that matters is the strongest one.** "Same hour, 7-day average" is both the
  best simple rule and the model's top feature, so the headline improvement is measured against it.
- **No day-of-year feature.** It was tried first: with a date split, every test date is a value
  the trees never saw, and validation error *rose* as trees were added. Recent history carries the
  season instead.

In [ ]:
f = json.load(open("docs/data/forecast.json"))
rows = {}
for target in ("departures", "arrivals"):
    for k, v in f[target]["hourly"].items():
        rows.setdefault(k, {})[f"{target} MAE"] = v["mae"]
rows["model"]["departures, daily MAE"] = f["departures"]["daily"]["model"]["mae"]
rows["seven_day_mean"]["departures, daily MAE"] = f["departures"]["daily"]["seven_day_mean"]["mae"]
table = pd.DataFrame(rows).T.loc[["model", "seven_day_mean", "yesterday", "seasonal_naive", "station_profile"]]
print("test months Nov-Dec; improvement vs the 7-day rule:", f"{f['improvement']['vs_seven_day_mean_hourly']:.1%} hourly,",
      f"{f['improvement']['vs_seven_day_mean_daily']:.1%} daily")
table.round(3)

In [ ]:
es = pd.DataFrame(f["departures"]["early_stopping"])
bt = pd.DataFrame(f["backtest"])
fig, (a, b) = plt.subplots(1, 2, figsize=(12, 3.6))
a.plot(es["trees"], es["valid_mae"], color=ORANGE, marker="o", ms=3)
a.set_title("Early stopping on Sep–Oct (departures)"); a.set_xlabel("trees"); a.set_ylabel("validation MAE")
for col, color, ls in (("model", ORANGE, "-"), ("seven_day_mean", BLUE, "-"), ("seasonal_naive", GRAY, "--")):
    b.plot(bt["month"], bt[col], color=color, ls=ls, marker="o", ms=3, label=col)
b.set_xticks(bt["month"], ["Aug", "Sep", "Oct", "Nov", "Dec"])
b.set_title("Rolling backtest: train on all earlier months"); b.legend(frameon=False, fontsize=8)
plt.tight_layout(); plt.show()
bt.assign(gain_vs_7d=1 - bt["model"] / bt["seven_day_mean"]).round(3)

**Uncertainty.** An 80% band from split-conformal calibration: the train-only model's Sep–Oct
predictions are binned by level, and each bin's 10th–90th percentile of actual trips becomes the
band for test hours predicted in that bin. Coverage on the test months:

In [ ]:
print(pd.Series(f["intervals"]))
ex = pd.DataFrame(f["example"]["series"]); ex["t"] = pd.to_datetime(ex["t"])
fig, ax = plt.subplots(figsize=(12, 3.6))
ax.fill_between(ex["t"], ex["p10"], ex["p90"], color=ORANGE, alpha=0.18, label="80% band")
ax.plot(ex["t"], ex["actual"], color=BLUE, lw=1.8, label="actual")
ax.plot(ex["t"], ex["pred"], color=ORANGE, lw=1.8, label="model")
ax.plot(ex["t"], ex["naive"], color=GRAY, lw=1.2, ls="--", label="same hour last week")
ax.set_title(f"{f['example']['station']}, week of {f['example']['week_start']} (last week was Thanksgiving)")
ax.legend(frameon=False, fontsize=8, ncol=4); plt.tight_layout(); plt.show()

**Does it help a dispatcher?** Rebalancing needs net flow, so the two models are combined into
each station's weekday 7–10am net flow and scored two ways: the error in that number, and how many
of each morning's 20 biggest losers the forecast puts in its own top 20.

In [ ]:
nf = f["net_flow_am"]
pd.DataFrame({
    "net-flow MAE (bikes per station-morning)": nf["mae"],
    f"top-{nf['top_k']} losers found (of {nf['top_k']})": {k: v * nf["top_k"] for k, v in nf["precision_losers"].items()},
}).round(2)

In [ ]:
imp = pd.DataFrame(f["feature_importances"]).set_index("label")["importance"].sort_values()
imp.plot.barh(color=BLUE, figsize=(6, 3.8), title="Permutation importance (departures, test sample)")
plt.tight_layout(); plt.show()

## 5. Takeaways and limitations

- The commute creates a large, **regular** imbalance: thousands of bikes every weekday morning,
  reversed every evening. Regular means plannable.
- Weather matters through **timing**: rush-hour rain costs about four times what off-peak rain does.
- The forecast beats the best simple rule overall and in every backtest month, and narrows
  net-flow error. It does **not** beat history at finding the very worst stations, which are the
  same every day; its value is on the rest of the network and on unusual days.

Limitations: no dock capacity or live station status (Citi Bike's GBFS feed would be the next
step), observed rather than forecast weather, station names as identifiers, and one year of data.